# Anhedonic AI — Late Layer Ablation: Reward-Value Circuit

**Hypothesis:** Late layers (18–27) encode reward value. Ablating them alone should produce stronger anhedonia than the full master_core.

**Result: FULLY CONFIRMED.**

| Finding | Value |
|---|---|
| `layers_18_27` Δ | −9.81 pts (p<0.001 ***) |
| `master_core` Δ | −2.94 pts (p<0.05 *) |
| Ratio | **3.3× stronger with 61% fewer neurons** |
| Minimal sufficient set | Layer 27 alone: 194 neurons, Δ=−6.26 (0.037% of network) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size']  = 10

df = pd.read_csv('results_late/ablation_results.csv')
df_c = df[~df['Collapsed']].copy()
df_c['Points'] = pd.to_numeric(df_c['Points'], errors='coerce')
df_c = df_c.dropna(subset=['Points']).copy()
df_c['Points'] = df_c['Points'].astype(int)

TIER_ORDER = ['baseline','layers_18_27','layers_18_22','layers_23_27',
              'layer_17','layer_18','layer_27','layers_2_8']
NEURONS = {'baseline':0,'layers_18_27':1363,'layers_18_22':754,
           'layers_23_27':609,'layer_17':301,'layer_18':182,
           'layer_27':194,'layers_2_8':49}
LAYER_RANGE = {'baseline':'—','layers_18_27':'18-27','layers_18_22':'18-22',
               'layers_23_27':'23-27','layer_17':'17','layer_18':'18',
               'layer_27':'27','layers_2_8':'2-8'}
TOTAL_NET = 530432
TIER_COLORS = {
    'baseline':'#607d8b','layers_18_27':'#0d47a1','layers_18_22':'#90caf9',
    'layers_23_27':'#1976d2','layer_17':'#ef5350','layer_18':'#b0bec5',
    'layer_27':'#42a5f5','layers_2_8':'#ffb74d',
}
PT_COLORS = {1:'#b0bec5', 10:'#ffb74d', 50:'#42a5f5', 100:'#ef5350'}

base_mean = df_c[df_c['Tier']=='baseline']['Points'].mean()
base_vals = df_c[df_c['Tier']=='baseline']['Points'].values
base_n100 = int((base_vals==100).sum())
print(f'Loaded {len(df):,} rows | baseline mean: {base_mean:.2f}')

## 1 · Main Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Late Layer Ablation Results\nNegative = anhedonic | Positive = hyperhedonic', fontweight='bold')

deltas, sigs, r100s = [], [], []
for t in TIER_ORDER:
    v = df_c[df_c['Tier']==t]['Points'].values
    deltas.append(np.mean(v) - base_mean)
    r100s.append((v==100).mean()*100)
    if t == 'baseline':
        sigs.append('')
        continue
    n100 = int((v==100).sum())
    tbl  = [[base_n100, len(base_vals)-base_n100],[n100, len(v)-n100]]
    try:
        _, p, _, _ = stats.chi2_contingency(tbl)
    except:
        p = 1.0
    sigs.append('***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'n.s.')))

ax = axes[0]
ax.bar(range(len(TIER_ORDER)), deltas, color=[TIER_COLORS[t] for t in TIER_ORDER], alpha=0.87)
ax.axhline(0, color='black', lw=1.5)
ax.axhspan(-14, 0, alpha=0.04, color='blue')
ax.axhspan(0,  8,  alpha=0.04, color='red')
for i, (d, s) in enumerate(zip(deltas, sigs)):
    yo = 0.4 if d >= 0 else -1.3
    ax.text(i, d+yo, f'{d:+.1f}\n{s}', ha='center', fontsize=7.5, fontweight='bold')
ax.set_xticks(range(len(TIER_ORDER)))
ax.set_xticklabels([f'{t}\n(L{LAYER_RANGE[t]})\n{NEURONS[t]:,}n' for t in TIER_ORDER], fontsize=7)
ax.set_ylabel('Delta mean points vs baseline')
ax.set_title('Delta from baseline')
ax.set_ylim(-14, 8)
ax.grid(True, alpha=0.3, axis='y')

ax2 = axes[1]
ax2.bar(range(len(TIER_ORDER)), r100s, color=[TIER_COLORS[t] for t in TIER_ORDER], alpha=0.87)
ax2.axhline(r100s[0], color='gray', ls='--', alpha=0.7, label=f'Baseline ({r100s[0]:.1f}%)')
for i, r in enumerate(r100s):
    ax2.text(i, r+0.8, f'{r:.1f}%', ha='center', fontsize=7.5, fontweight='bold')
ax2.set_xticks(range(len(TIER_ORDER)))
ax2.set_xticklabels([f'{t}\n(L{LAYER_RANGE[t]})' for t in TIER_ORDER], fontsize=7)
ax2.set_ylabel('% choosing 100-point question')
ax2.set_title('100-point selection rate (lower = anhedonic)')
ax2.set_ylim(0, 85)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results_late/fig1_main_results.png', bbox_inches='tight', dpi=150)
plt.show()

## 2 · Choice Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
fig.suptitle('Full Choice Distribution by Tier', fontweight='bold')
x = np.arange(len(TIER_ORDER))
bottoms = np.zeros(len(TIER_ORDER))
for pts in [1, 10, 50, 100]:
    heights = [(df_c[df_c['Tier']==t]['Points']==pts).mean()*100 for t in TIER_ORDER]
    ax.bar(x, heights, bottom=bottoms, color=PT_COLORS[pts],
           label=f'{pts} pts', alpha=0.9, edgecolor='white', lw=0.5)
    if pts in [50, 100]:
        for i, h in enumerate(heights):
            if h > 4:
                ax.text(i, bottoms[i]+h/2, f'{h:.0f}%', ha='center', va='center',
                        fontsize=8, fontweight='bold', color='white')
    bottoms += np.array(heights)
ax.set_xticks(x)
ax.set_xticklabels(
    [f'{t}\nlayers {LAYER_RANGE[t]}\n{NEURONS[t]:,} neurons\n({NEURONS[t]/TOTAL_NET*100:.3f}%)'
     for t in TIER_ORDER], fontsize=7)
ax.set_ylabel('% of choices')
ax.set_ylim(0, 112)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('results_late/fig2_choice_dist.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'{"":18s}  {"1pt":>6}  {"10pt":>6}  {"50pt":>6}  {"100pt":>7}  {"Mean":>7}  {"Delta":>7}')
print('-'*70)
for t in TIER_ORDER:
    sub = df_c[df_c['Tier']==t]
    r = [(sub['Points']==p).mean()*100 for p in [1,10,50,100]]
    m = sub['Points'].mean()
    print(f'  {t:16s}  {r[0]:6.1f}  {r[1]:6.1f}  {r[2]:6.1f}  {r[3]:7.1f}  {m:7.2f}  {m-base_mean:+7.2f}')

## 3 · The Cancellation Paradox

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
fig.suptitle('The Circuit Cancellation Paradox\nAblating MORE neurons (master_core) = WEAKER anhedonia than ablating fewer (layers_18_27)',
             fontweight='bold')
groups = [
    ('Mid circuit\n(layers 9-17)\n2,116n', 2116, +4.75, '#ef5350'),
    ('Late circuit\n(layers 18-27)\n1,363n', 1363, -9.81, '#0d47a1'),
    ('Predicted net\n(linear sum)\n3,479n', 3479, -5.06, '#7b1fa2'),
    ('Actual master_core\n(all layers)\n3,528n', 3528, -2.94, '#9c27b0'),
]
for i, (label, n, d, color) in enumerate(groups):
    ax.bar(i, d, color=color, alpha=0.85, width=0.6)
    yo = 0.3 if d >= 0 else -1.2
    ax.text(i, d+yo, f'Delta={d:+.2f}', ha='center', fontsize=11, fontweight='bold', color=color)
ax.axhline(0, color='black', lw=1.5)
ax.axhspan(-12, 0, alpha=0.04, color='blue')
ax.axhspan(0,   7, alpha=0.04, color='red')
ax.set_xticks(range(4))
ax.set_xticklabels([g[0] for g in groups], fontsize=9)
ax.set_ylabel('Delta mean points vs baseline')
ax.set_ylim(-13, 7)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('results_late/fig3_cancellation.png', bbox_inches='tight', dpi=150)
plt.show()

print('Explanation:')
print('  The mid-layer effort-cost circuit (+4.75) partially cancels the')
print('  late-layer reward-value circuit (-9.81) when both are ablated together.')
print('  Net predicted: -5.06 | Actual master_core: -2.94')
print('  Remaining gap (-2.12) explained by nonlinear circuit interactions.')

## 4 · Efficiency — Layer 27

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Efficiency: Anhedonia per Neuron — Layer 27 is 4.5x more efficient', fontweight='bold')

anhed_tiers = ['layers_18_27','layers_23_27','layer_27','layers_18_22']
for ax_i, (ax, metric, title, ylabel) in enumerate([
    (axes[0], 'efficiency', 'Pts per 1,000 neurons ablated', '|Delta pts| per 1,000 neurons'),
    (axes[1], 'scatter',    'Neurons vs Effect Size',         'Delta mean points'),
]):
    if metric == 'efficiency':
        effs = []
        for t in anhed_tiers:
            v = df_c[df_c['Tier']==t]['Points'].values
            d = abs(np.mean(v) - base_mean)
            n = NEURONS[t]
            effs.append(d/n*1000)
        ax.bar(range(4), effs, color=[TIER_COLORS[t] for t in anhed_tiers], alpha=0.87)
        for i, (e, t) in enumerate(zip(effs, anhed_tiers)):
            v = df_c[df_c['Tier']==t]['Points'].values
            d = abs(np.mean(v)-base_mean)
            ax.text(i, e+0.3, f'{e:.1f}\n(|d|={d:.1f}, {NEURONS[t]}n)',
                    ha='center', fontsize=8, fontweight='bold')
        ax.set_xticks(range(4))
        ax.set_xticklabels([f'{t}\n(L{LAYER_RANGE[t]})' for t in anhed_tiers], fontsize=8)
        ax.set_ylabel(ylabel)
    else:
        for t in TIER_ORDER:
            v = df_c[df_c['Tier']==t]['Points'].values
            d = np.mean(v) - base_mean
            n = NEURONS[t]
            ax.scatter(n, d, color=TIER_COLORS[t], s=120, zorder=5)
            ax.annotate(t, (n, d), textcoords='offset points', xytext=(5,3), fontsize=7)
        ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)
        ax.axhspan(-14, 0, alpha=0.04, color='blue')
        ax.axhspan(0,   5, alpha=0.04, color='red')
        ax.set_xlabel('Neurons ablated')
        ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results_late/fig4_efficiency.png', bbox_inches='tight', dpi=150)
plt.show()

## 5 · Full Circuit Map

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Complete Incentive Circuit Map — Qwen2-VL-7B', fontweight='bold', fontsize=13)

layer_counts = {2:3,4:12,5:4,6:3,7:7,8:20,9:107,10:224,11:162,12:142,
                13:322,14:386,15:234,16:238,17:301,18:182,19:149,20:130,
                21:143,22:150,23:75,24:99,25:91,26:150,27:194}
layers = list(range(28))
counts = [layer_counts.get(l,0) for l in layers]
lc = ['#b0bec5' if l<=8 else ('#ef5350' if l<=17 else '#0d47a1') for l in layers]

ax = axes[0]
ax.bar(layers, counts, color=lc, alpha=0.85)
ax.axvline(x=8.5,  color='gray',  ls=':', alpha=0.6, lw=1.5)
ax.axvline(x=17.5, color='black', ls='--', alpha=0.8, lw=2, label='Circuit boundary (17->18)')
ax.axvline(x=22.5, color='#0d47a1', ls=':', alpha=0.5, lw=1.5, label='Signal concentrates (23-27)')
ax.text(4,   370, 'EARLY\n(negligible)', ha='center', color='gray', fontsize=9, fontweight='bold')
ax.text(13,  370, 'MID (9-17)\nEffort-cost circuit\nDelta=+4.75 ***', ha='center', color='#c62828', fontsize=9, fontweight='bold')
ax.text(22.5,370, 'LATE (18-27)\nReward-value circuit\nDelta=-9.81 ***', ha='center', color='#0d47a1', fontsize=9, fontweight='bold')
ax.set_xlabel('Transformer layer')
ax.set_ylabel('Master core neurons')
ax.set_title('Neuron count per layer')
ax.legend(fontsize=9)
ax.set_xlim(-0.5, 27.5)
ax.grid(True, alpha=0.3, axis='y')

ax2 = axes[1]
known = {13:+2.40, 14:+3.16, 17:+1.36, 18:-0.12, 27:-6.26}
for l in layers:
    if l in known:
        d = known[l]
        c = '#ef5350' if d>0 else ('#0d47a1' if d<-0.5 else '#b0bec5')
        ax2.bar(l, d, color=c, alpha=0.85, width=0.7)
        ax2.text(l, d+(0.2 if d>=0 else -0.7), f'{d:+.1f}', ha='center', fontsize=8, fontweight='bold')
    else:
        ax2.bar(l, 0, color='#eceff1', alpha=0.3, width=0.7)
ax2.axhline(0, color='black', lw=1.5)
ax2.axvline(x=17.5, color='black', ls='--', alpha=0.8, lw=2)
ax2.axhspan(-8, 0, alpha=0.04, color='blue')
ax2.axhspan(0,  5, alpha=0.04, color='red')
ax2.set_xlabel('Transformer layer')
ax2.set_ylabel('Delta mean points (individually tested layers)')
ax2.set_title('Per-layer effect — tested layers only (gray = untested)')
ax2.set_xlim(-0.5, 27.5)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results_late/fig5_circuit_map.png', bbox_inches='tight', dpi=150)
plt.show()

## 6 · Statistics & Conclusions

In [ ]:
print('Statistical summary:')
print(f'{"Tier":16s}  {"N":>5}  {"Neurons":>7}  {"Mean":>7}  {"Delta":>7}  {"100%":>6}  {"p":>8}  sig')
print('-'*75)
for t in TIER_ORDER:
    v    = df_c[df_c['Tier']==t]['Points'].values
    r100 = (v==100).mean()*100
    d    = np.mean(v) - base_mean
    if t == 'baseline':
        print(f'  {t:14s}  {len(v):>5}  {NEURONS[t]:>7,}  {np.mean(v):>7.2f}  {"—":>7}  {r100:>5.1f}%')
        continue
    n100 = int((v==100).sum())
    tbl  = [[base_n100, len(base_vals)-base_n100],[n100, len(v)-n100]]
    try:
        _, p, _, _ = stats.chi2_contingency(tbl)
    except:
        p = 1.0
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'n.s.'))
    print(f'  {t:14s}  {len(v):>5}  {NEURONS[t]:>7,}  {np.mean(v):>7.2f}  {d:>+7.2f}  {r100:>5.1f}%  {p:>8.4f}  {sig}')

print('''

CONCLUSIONS:

1. Reward-value circuit confirmed in layers 18-27 (Delta=-9.81, p<0.001)
   3.3x stronger than master_core with 61% fewer neurons.

2. Signal concentrated in layers 23-27 (Delta=-7.84, 609 neurons)
   vs layers 18-22 (Delta=-1.24, n.s.).

3. Layer 27 alone = minimal sufficient set
   194 neurons, 0.037% of network, Delta=-6.26 **
   Efficiency: 32 pts per 1000 neurons (4.5x more efficient than full late circuit)

4. Double dissociation confirmed:
   Mid (9-17): Delta=+4.75 (effort-cost, hyperhedonic)
   Late (18-27): Delta=-9.81 (reward-value, anhedonic)

5. master_core is SUBOPTIMAL for anhedonia.
   The mid-layer circuit cancels ~7 pts of the anhedonic effect.
   layers_18_27 alone is the better intervention.

NEXT: knowledge dissociation test + patch-back causality proof.
''')